In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle as pkl
from torch.utils.data import Dataset, DataLoader
import torch

In [2]:
with open ('/home/guleserhocam/VS_Projects/simglucose/dataset/Datad4rl/hopper-medium-v2.pkl', 'rb') as obj:
    df = pkl.load(obj)

In [19]:
idx = 5
print(len(df[idx]['next_observations']))
print(len(df[idx]['observations']))
print(len(df[idx]['actions']))
print(len(df[idx]['rewards']))
print(len(df[idx]['terminals']))
print(f"Last terminal state: {df[idx]['terminals'][-1]}")

415
415
415
415
415
Last terminal state: True


In [3]:
#Check data types under dataset
data = df
print(type(data))
if isinstance(data, list):
    print(type(data[0]))
if isinstance(data, dict):
    print(data.keys())

<class 'list'>
<class 'dict'>


In [9]:
class CustomDataset(Dataset):
    def __init__(self, data, max_len):
        self.obs = [item['observations'][:max_len] for item in data]
        self.nex_obs = [item['next_observations'][:max_len] for item in data]
        self.actions = [item['actions'][:max_len] for item in data]
        self.rewards = [item['rewards'][:max_len] for item in data]
        self.terminals = [item['terminals'][:max_len] for item in data]

    def __len__(self):
        return len(self.obs)

    def __getitem__(self, idx):
        s = torch.tensor(self.obs[idx], dtype=torch.float)
        n = torch.tensor(self.nex_obs[idx], dtype=torch.float)
        a = torch.tensor(self.actions[idx], dtype=torch.float)
        r = torch.tensor(self.rewards[idx], dtype=torch.float)
        d = torch.tensor(self.terminals[idx], dtype=torch.float)
        sample = {"observations": s, "next_observations": n, "actions":a, "rewards": r, "terminals":d}
        return sample


In [10]:
max_len = 100
df_new = CustomDataset(df, max_len)
print(f'Length of data:{len(df_new)}')
sample = df_new[0]  # Equivalent to dataset.__getitem__(0)

print("Sample at index 0:")
print("Observations:", sample["observations"])
print("Next Observations:", sample["next_observations"])
print("Actions:", sample["actions"])
print("Rewards:", sample["rewards"])
print("Terminals:", sample["terminals"])

Length of data:2186
Sample at index 0:
Observations: tensor([[ 1.2499e+00,  1.9746e-03,  5.2147e-05,  ...,  2.3277e-03,
          3.1283e-03,  1.2418e-03],
        [ 1.2496e+00,  5.5432e-04,  5.0323e-04,  ...,  9.1810e-02,
         -9.5989e-01, -1.0461e-01],
        [ 1.2488e+00, -4.3362e-03,  1.0789e-03,  ...,  5.2861e-02,
         -1.9027e+00, -2.7964e-01],
        ...,
        [ 1.1981e+00,  1.1557e-02, -6.0221e-01,  ...,  1.3407e+00,
         -4.5000e-01, -9.7017e+00],
        [ 1.2175e+00,  1.6895e-02, -5.8729e-01,  ...,  2.3894e+00,
         -7.6395e-01, -1.0000e+01],
        [ 1.2380e+00,  2.3782e-02, -5.6808e-01,  ...,  2.4158e+00,
         -4.0172e-01, -1.0000e+01]])
Next Observations: tensor([[ 1.2496e+00,  5.5432e-04,  5.0323e-04,  ...,  9.1810e-02,
         -9.5989e-01, -1.0461e-01],
        [ 1.2488e+00, -4.3362e-03,  1.0789e-03,  ...,  5.2861e-02,
         -1.9027e+00, -2.7964e-01],
        [ 1.2475e+00, -1.2123e-02,  1.3424e-03,  ...,  1.6898e-02,
         -2.3135e+00, -

In [11]:
from torch.nn.utils.rnn import pad_sequence

def my_collate_fn(batch):
    obs = torch.tensor([item['observations'] for item in batch])
    nex_obs = torch.tensor([item['next_observations'] for item in batch])
    actions = torch.tensor([item['actions'] for item in batch])
    rewards = torch.tensor([item['rewards'] for item in batch])
    terminals = torch.tensor([item['terminals'] for item in batch])

    # Pad observations and next_observations to the same length
    #obs_padded = pad_sequence(obs, batch_first=True)
    #print(obs.shape, obs_padded.shape)
    #nex_obs_padded = pad_sequence(nex_obs, batch_first=True)

    return {
        "observations": obs,
        "next_observations": nex_obs,
        "actions": actions,
        "rewards": rewards,
        "terminals": terminals,
    }

In [12]:
# Create a DataLoader
#dataloader = DataLoader(df_new, batch_size=5, shuffle=True, collate_fn=my_collate_fn)
dataloader = DataLoader(df_new, batch_size=5, shuffle=True)

# Iterate through the DataLoader
for batch_idx, batch in enumerate(dataloader):
    print(f"Batch {batch_idx}:")
    print("Observations:", batch["observations"].shape)
    print("Actions:", batch["actions"].shape)
    print("Rewards:", batch["rewards"].shape)

Batch 0:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 1:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 2:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 3:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 4:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 5:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 6:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 7:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
Batch 8:
Observations: torch.Size([5, 100, 11])
Actions: torch.Size([5, 100, 3])
Rewards: torch.Size([5, 100])
B